# PyG molecular regression (notebooks)

Copy any **code** cell into another notebook, or run this file from the repo with the kernel’s working directory set to the **project root** (folder that contains `src/` and `pyproject.toml`).

Requires: `pip install openadnet[dl]`.

## 1. Put `src` on `sys.path` (paste at top of a notebook)

Run once per session if imports like `models` fail.

In [1]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
if (_root / "src").is_dir():
    sys.path.insert(0, str(_root / "src"))
else:
    raise RuntimeError("Start the kernel cwd from the openadnet repo (or folder containing pyproject.toml)")

## 2. K-fold CV (full data)

Same idea as `scripts/cv_gnn_regressor.py`. Change `architecture` to `gcn`, `gat`, `mpnn`, `graphconv`, or `attentivefp`.

In [3]:
from IPython.display import display

from baseline import BaselineCVConfig
from load_data import train
from models.cv_dl import run_gnn_regressor_cv

cfg = BaselineCVConfig(n_splits=3, shuffle=True, cv_random_state=0, y_col="pEC50")
fold_df, summary = run_gnn_regressor_cv(
    train,
    smiles_col="SMILES",
    target_cols=["pEC50"],
    architecture="gin",
    config=cfg,
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    hidden_dim=64,
    num_layers=3,
    gat_heads=4,
    show_progress=True,
    fit_show_progress=False,
)
display(fold_df)
display(summary)

gnn_cv:   0%|          | 0/3 [00:00<?, ?fold/s]

,fold,rmse,mae,r2,n_train,n_val
0,0,0.817705,0.643558,0.437740,2760,1380
1,1,0.823294,0.623762,0.464761,2760,1380
2,2,0.896853,0.667897,0.388807,2760,1380


mean_rmse    0.845951
std_rmse     0.036066
mean_mae     0.645072
std_mae      0.018050
mean_r2      0.430436
std_r2       0.031435
dtype: float64

## 2b. K-fold CV with baseline descriptors on nodes (optional)

Same as §2, but pass **`descriptor_name`** so each graph uses **`build_descriptor_matrix`** (joblib cache under `OPENADNET_FP_CACHE`) and **broadcasts** that vector to every atom. **MPNN** uses these via `edge_attr` plus the wider node vector.

Use a **moderate** fingerprint width (e.g. `morgan_r2_bits_512`) unless you have GPU memory to spare—very large dims on every node are expensive.

In [ ]:
from IPython.display import display

from baseline import BaselineCVConfig
from load_data import train
from models.cv_dl import run_gnn_regressor_cv

cfg = BaselineCVConfig(n_splits=3, shuffle=True, cv_random_state=0, y_col="pEC50")
fold_df, summary = run_gnn_regressor_cv(
    train,
    smiles_col="SMILES",
    target_cols=["pEC50"],
    architecture="mpnn",
    descriptor_name="morgan_r2_bits_512",
    config=cfg,
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    hidden_dim=64,
    num_layers=2,
    gat_heads=4,
    show_progress=True,
    fit_show_progress=False,
)
display(fold_df)
display(summary)

In [5]:
fold_df, summary = run_gnn_regressor_cv(
    train,
    smiles_col="SMILES",
    target_cols=["pEC50"],
    architecture="gat",
    config=cfg,
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    hidden_dim=64,
    num_layers=2,
    gat_heads=4,
    show_progress=True,
    fit_show_progress=False,
)
display(fold_df)
display(summary)

gnn_cv:   0%|          | 0/3 [00:00<?, ?fold/s]

,fold,rmse,mae,r2,n_train,n_val
0,0,0.882515,0.655109,0.345080,2760,1380
1,1,0.847200,0.637606,0.433226,2760,1380
2,2,0.916785,0.674378,0.361339,2760,1380


mean_rmse    0.882167
std_rmse     0.028409
mean_mae     0.655698
std_mae      0.015018
mean_r2      0.379881
std_r2       0.038300
dtype: float64

In [ ]:
fold_df, summary = run_gnn_regressor_cv(
    train,
    smiles_col="SMILES",
    target_cols=["pEC50"],
    architecture="mpnn",
    config=cfg,
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    hidden_dim=64,
    num_layers=2,
    gat_heads=4,
    show_progress=True,
    fit_show_progress=False,
)
display(fold_df)
display(summary)

gnn_cv:   0%|          | 0/3 [00:00<?, ?fold/s]

## 3. K-fold CV (small subset — fast)

Mirrors `examples/quick_cv_gnn_subset.py`.

In [ ]:
from baseline import BaselineCVConfig
from load_data import train
from models.cv_dl import run_gnn_regressor_cv

small = train.head(300).copy()
cfg = BaselineCVConfig(n_splits=3, shuffle=True, cv_random_state=42, y_col="pEC50")
fold_df, summary = run_gnn_regressor_cv(
    small,
    smiles_col="SMILES",
    target_cols=["pEC50"],
    architecture="mpnn",
    config=cfg,
    epochs=2,
    batch_size=32,
    learning_rate=1e-3,
    hidden_dim=48,
    num_layers=2,
    gat_heads=4,
    show_progress=True,
)

## 4. Single train / validation split (holdout)

Same idea as `examples/holdout_regression.py --backend gnn`.

In [35]:
from load_data import train
from models.cv_dl import prepare_regression_frame, regression_metrics
from models.data import graph_regression_from_dataframe
from models.data.graph import train_val_split_graph
from models.nn.pyg_regressor import PyGMoleculeRegressor

work = prepare_regression_frame(train, "SMILES", ["pEC50"])
full_ds = graph_regression_from_dataframe(work, "SMILES", ["pEC50"])
tr_ds, va_ds = train_val_split_graph(full_ds, val_fraction=0.15, random_state=0)

model = PyGMoleculeRegressor(
    n_tasks=1,
    architecture="gcn",
    hidden_dim=64,
    num_layers=3,
    gat_heads=4,
)
model.fit(
    tr_ds,
    epochs=10,
    batch_size=32,
    learning_rate=1e-3,
    val_dataset=va_ds,
    show_progress=True,
)
pred = model.predict(va_ds, show_progress=False)
regression_metrics(va_ds.y, pred)

epoch: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:44<00:00,  4.46s/it, loss=0.721, lr=0.001, val_loss=0.704]


{'rmse': 0.8368297944173908,
 'mae': 0.6403267686071411,
 'r2': 0.4006816004869811}

## 5. `GNNRegressor` — GAT default, other `architecture=` values

Convenience subclass with default **`architecture="gat"`**. Pass `architecture="gin"` (or any registry name) to match `PyGMoleculeRegressor`.

In [ ]:
from models import GNNRegressor

model = GNNRegressor(n_tasks=1, hidden_dim=64, num_layers=3)
# same as PyGMoleculeRegressor(..., architecture="gat")
# GNNRegressor(..., architecture="gin") for the previous default encoder

## 6. Architecture names (for `architecture=`)

`gat` and `attentivefp` use `gat_heads`.

In [ ]:
from models.nn.registry import ARCHITECTURES

list(ARCHITECTURES)

## 7. Optional: force CPU

Use if CUDA errors in notebooks.

In [ ]:
import os
import torch

os.environ.setdefault("OPENADNET_FORCE_CPU", "1")
device = torch.device("cpu")
# PyGMoleculeRegressor(..., device=device)

## 8. Baseline fingerprint / descriptor on each node (cached)

Uses `features_data.build_descriptor_matrix` (same joblib cache as sklearn baselines). The molecule-level vector is **broadcast** to every atom and concatenated to `x`. Pass **`descriptor_name`** as a **string** or a **tuple/list of names** to concatenate descriptors in order (same `desc` for `graph_regression_from_dataframe` and `PyGMoleculeRegressor`). Use **`descriptor_dim_for_names(desc)`** for the total extra width. Prefer smaller fps (e.g. `morgan_r2_bits_512`) or `rdkit_phys_props` to save memory.

See [docs/pyg_gnn.md](../docs/pyg_gnn.md) and [`examples/notebook_graph_with_descriptors.py`](../examples/notebook_graph_with_descriptors.py).

In [1]:
from models.cv_dl import prepare_regression_frame, regression_metrics
from load_data import train
from models.data import graph_regression_from_dataframe
from models.nn.pyg_regressor import PyGMoleculeRegressor
from features_data import descriptor_dim_for_names, list_descriptor_names
from sklearn.model_selection import train_test_split
list(list_descriptor_names())


['morgan_r0_bits_512',
 'morgan_r0_count_512',
 'morgan_r1_bits_512',
 'morgan_r1_count_512',
 'morgan_r2_bits_512',
 'morgan_r2_count_512',
 'rdkit_bits_512',
 'rdkit_count_512',
 'atom_pair_bits_512',
 'atom_pair_count_512',
 'torsion_bits_512',
 'torsion_count_512',
 'morgan_r0_bits_1024',
 'morgan_r0_count_1024',
 'morgan_r1_bits_1024',
 'morgan_r1_count_1024',
 'morgan_r2_bits_1024',
 'morgan_r2_count_1024',
 'rdkit_bits_1024',
 'rdkit_count_1024',
 'atom_pair_bits_1024',
 'atom_pair_count_1024',
 'torsion_bits_1024',
 'torsion_count_1024',
 'morgan_r0_bits_2048',
 'morgan_r0_count_2048',
 'morgan_r1_bits_2048',
 'morgan_r1_count_2048',
 'morgan_r2_bits_2048',
 'morgan_r2_count_2048',
 'rdkit_bits_2048',
 'rdkit_count_2048',
 'atom_pair_bits_2048',
 'atom_pair_count_2048',
 'torsion_bits_2048',
 'torsion_count_2048',
 'rdkit_phys_props']

In [28]:
# Pass one name (str) or several (tuple/list); vectors are concatenated in that order on every atom.
desc = "morgan_r1_count_512"
# desc = ("rdkit_phys_props", "morgan_r1_count_512")
desc = "rdkit_phys_props"
print("extra node dims (descriptor block only):", descriptor_dim_for_names(desc))

# # small = train.head(100)
# ds = graph_regression_from_dataframe(
#     train, "SMILES", ["pEC50"], descriptor_name=desc
# )
# model = PyGMoleculeRegressor(
#     1, architecture="gat", descriptor_name=desc, hidden_dim=48, num_layers=2
# )
# print(model)
# model.fit(ds, epochs=100, batch_size=64, show_progress=True)


extra node dims (descriptor block only): 15


In [47]:
tr_df, rest = train_test_split(train, test_size=0.2, random_state=0)
val_df, te_df = train_test_split(rest, test_size=0.5, random_state=0)  # 10% val, 10% test

train_ds = graph_regression_from_dataframe(
    tr_df, "SMILES", ["pEC50"], descriptor_name=desc
)
val_ds = graph_regression_from_dataframe(
    val_df, "SMILES", ["pEC50"], descriptor_name=desc
)
test_ds = graph_regression_from_dataframe(
    te_df, "SMILES", ["pEC50"], descriptor_name=desc
)

model = PyGMoleculeRegressor(
    1, architecture="gin", descriptor_name=desc, hidden_dim=128, num_layers=5
)
model.fit(
    train_ds,
    epochs=50,
    batch_size=4 * 128,
    learning_rate=0.001,
    val_dataset=val_ds,
    show_progress=True,
)

epoch:  64%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                                                          | 32/50 [01:42<00:56,  3.16s/it, loss=0.532, lr=0.001, val_loss=0.59]

val plateau: reducing lr to 5.00e-04 (1/5)


epoch:  84%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                | 42/50 [02:14<00:25,  3.16s/it, loss=0.442, lr=0.0005, val_loss=0.595]

val plateau: reducing lr to 2.50e-04 (2/5)


epoch: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [02:40<00:00,  3.20s/it, loss=0.406, lr=0.00025, val_loss=0.578]


[18.774996076311385,
 13.56104006086077,
 10.393417085920062,
 7.988262380872454,
 6.306333678109305,
 4.928969655718122,
 3.6889897755214145,
 2.7877022198268344,
 2.0951777696609497,
 1.5224661486489433,
 1.177074636731829,
 0.9259099960327148,
 0.7844263911247253,
 0.7136187042508807,
 0.673236438206264,
 0.6442792245319912,
 0.6282997812543597,
 0.6182264515331813,
 0.6630952869142804,
 0.621990944658007,
 0.6014515502112252,
 0.5858957512038094,
 0.5843068276132856,
 0.5743267025266375,
 0.5751462408474514,
 0.564866874899183,
 0.5515494006020683,
 0.5628031662532261,
 0.5452484999384198,
 0.5355231804507119,
 0.5473938158580235,
 0.5319235920906067,
 0.5153902981962476,
 0.49043164934430805,
 0.47524691053799223,
 0.48080776418958393,
 0.45602789521217346,
 0.47064277103969027,
 0.4496511476380484,
 0.4574481461729322,
 0.4418323721204485,
 0.4418726180280958,
 0.43541681340762545,
 0.4246410386902945,
 0.4308322327477591,
 0.42594500950404574,
 0.4230039119720459,
 0.40748395238

In [48]:
y_pred = model.predict(test_ds, batch_size=64, show_progress=False)
y_true = test_ds.y  # same row order as predict

metrics = regression_metrics(y_true, y_pred)
print(metrics)  # {"rmse": ..., "mae": ..., "r2": ...}

{'rmse': 0.8162010016863009, 'mae': 0.5942563250214582, 'r2': 0.44675007412551737}


In [38]:
y_pred = model.predict(test_ds, batch_size=64, show_progress=False)
y_true = test_ds.y  # same row order as predict

metrics = regression_metrics(y_true, y_pred)
print(metrics)  # {"rmse": ..., "mae": ..., "r2": ...}

{'rmse': 0.8309142561491428, 'mae': 0.6085068112755743, 'r2': 0.4266239635760568}


In [8]:
y_pred = model.predict(test_ds, batch_size=64, show_progress=False)
y_true = test_ds.y  # same row order as predict

metrics = regression_metrics(y_true, y_pred)
print(metrics)  # {"rmse": ..., "mae": ..., "r2": ...}

{'rmse': 0.9818975531537161, 'mae': 0.747348151230006, 'r2': 0.19931909177535223}


In [12]:
y_pred = model.predict(test_ds, batch_size=64, show_progress=False)
y_true = test_ds.y  # same row order as predict

metrics = regression_metrics(y_true, y_pred)
print(metrics)  # {"rmse": ..., "mae": ..., "r2": ...}

{'rmse': 0.775618280388314, 'mae': 0.5748637353394919, 'r2': 0.4942290096450507}
